# TPI · Módulo 3 — Extracción de descriptores
**PSIB 2026Q2 · Grupo 12** — Lautersztein, Reggio, Ballester
*Detección de leucemia linfoblástica aguda (LLA) en imágenes de frotis de sangre periférica*

A partir de la máscara del núcleo del Módulo 2 calculamos números que describen cada célula, para después (Módulo 4) buscar cuáles separan las imágenes benignas de las patológicas. Según el anteproyecto, los linfoblastos tienen **núcleos más grandes, irregulares o hendidos, con alta relación núcleo/citoplasma y cromatina fina**, mientras que los linfocitos maduros y las hematogonias tienen núcleos más chicos, redondos y con cromatina densa.

El notebook se arma por partes:
1. **Descriptores de forma** (área, perímetro, circularidad, excentricidad, solidez, relación núcleo/célula). ← *esta parte*
2. Descriptores de color (H, S, a\*, b\* dentro del núcleo).
3. Descriptores de textura (GLCM y LBP).
4. Efecto de la compresión JPEG sobre los descriptores.

La salida es una tabla con una fila por imagen y una columna por descriptor (`modulo3_descriptores.csv`), que usa el Módulo 4.

## 0. Carga y funciones de los módulos anteriores
Mismo índice del dataset que en los Módulos 1 y 2, y las funciones ya elegidas: corrección de iluminación por **cierre morfológico** (Módulo 1) y segmentación del núcleo por **Otsu sobre S** dentro de la célula (Módulo 2). Se copian acá para que el notebook se pueda correr solo.

In [ ]:
import os, glob
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
from skimage import measure
import kagglehub

RUTA = kagglehub.dataset_download('mehradaria/leukemia')
df = pd.DataFrame({'ruta': sorted(glob.glob(f'{RUTA}/**/*.jp*g', recursive=True))})
df['tipo'] = np.where(df.ruta.str.contains('segment', case=False), 'segmentada', 'original')
df['subtipo'] = df.ruta.map(lambda p: os.path.basename(os.path.dirname(p)))
df['clase'] = np.where(df.subtipo.str.contains('benign', case=False), 'no patológica', 'patológica')
df['archivo'] = df.ruta.map(os.path.basename)

org = df[df.tipo == 'original'].reset_index(drop=True)
seg = df[df.tipo == 'segmentada'].set_index(['subtipo', 'archivo']).ruta
org['ruta_seg'] = org.set_index(['subtipo', 'archivo']).index.map(seg.get)
org = org[org.ruta_seg.notna()].reset_index(drop=True)
leer = lambda ruta: cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)
print(f'Imágenes con referencia: {len(org)}')

In [ ]:
# --- Módulo 1: corrección de iluminación (cierre morfológico) ---
TAM_EE = 1 / 3

def estimar_campo(v):
    n = max(v.shape)
    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(n * TAM_EE) | 1,) * 2)
    return cv2.GaussianBlur(cv2.morphologyEx(v, cv2.MORPH_CLOSE, ee), (0, 0), n / 16)

def preprocesar(rgb):
    h, s, v = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV))
    campo = np.maximum(estimar_campo(v.astype(np.float32)), 1)
    v = np.clip(v / campo * campo.mean(), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2RGB)

# --- Módulo 2: ROI de la célula + segmentación del núcleo (Otsu en S) ---
AREA_MIN = 25

def mascara_celula(ruta_seg):
    return cv2.cvtColor(leer(ruta_seg), cv2.COLOR_RGB2GRAY) > 10

def limpiar(mask, area_min=AREA_MIN):
    m = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, etq, stats, _ = cv2.connectedComponentsWithStats(m)
    grandes = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= area_min]
    return np.isin(etq, grandes)

def segmentar(rgb, roi):
    s = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 1]
    umbral, _ = cv2.threshold(s[roi], 0, 255, cv2.THRESH_OTSU)
    return limpiar((s > umbral) & roi)

## 1. Descriptores de forma
### 1.0 Diagnóstico: ¿la segmentación de referencia sirve en todas las imágenes?
Al mirar los ejemplos vimos dos cosas:
- Cada imagen tiene **varias células** (no una sola), y la segmentación de referencia marca todas, además de **restos chicos** del frotis.
- En algunas imágenes de **Pro-B** la referencia viene **vacía** aunque se ven células violetas claras. Como el núcleo se busca *dentro* de la referencia, ahí no se detecta nada.

Para cuantificarlo medimos, por subtipo, qué fracción de la imagen ocupa la referencia y cuántas imágenes la tienen prácticamente vacía (< 0,5 % de la imagen).

In [ ]:
org['roi_frac'] = [mascara_celula(r).mean() for r in org.ruta_seg]
diag = org.groupby('subtipo').roi_frac.agg(
    mediana='median',
    vacias=lambda x: (x < 0.005).sum(),
    pct_vacias=lambda x: 100 * (x < 0.005).mean())
diag.round(3)

### 1.1 Medición núcleo por núcleo
Como hay varios núcleos por imagen, cada componente conexa de la máscara se mide **por separado** con `skimage.measure.regionprops`, descartando los componentes más chicos que `AREA_MIN_NUC` (restos y ruido, que además daban circularidades > 1 imposibles). Después se resume cada imagen con la **mediana** de cada descriptor sobre sus núcleos (la mediana no se deja arrastrar por un núcleo mal segmentado).

| Descriptor | Qué mide | Qué esperamos en LLA |
|---|---|---|
| `area` | tamaño del núcleo [px²] | mayor |
| `perimetro` | largo del contorno [px] | mayor |
| `circularidad` | $4\pi A / P^2$: 1 = círculo perfecto, baja si el borde es irregular | menor |
| `excentricidad` | 0 = círculo, →1 = alargado (de la elipse equivalente) | — |
| `solidez` | área / área de la envolvente convexa: < 1 si hay hendiduras | menor |
| `rel_nc` | área total de núcleos / área de las células (ROI) | mayor |
| `n_nucleos` | cantidad de núcleos detectados en la imagen | mayor (más blastos) |

In [ ]:
AREA_MIN_NUC = 100     # px²: debajo de esto es resto/ruido (meseta plana del histograma de 1.3); los núcleos forman el pico de ~300 px²
FORMA = ['area', 'perimetro', 'circularidad', 'excentricidad', 'solidez', 'rel_nc', 'n_nucleos']

def medir_nucleos(nucleo):
    # Una fila por núcleo (componente conexa con área >= AREA_MIN_NUC).
    filas = []
    for r in measure.regionprops(measure.label(nucleo)):
        if r.area < AREA_MIN_NUC:
            continue
        p = r.perimeter
        filas.append(dict(area=r.area, perimetro=p,
                          circularidad=4 * np.pi * r.area / p**2 if p > 0 else np.nan,
                          excentricidad=r.eccentricity, solidez=r.solidity))
    return pd.DataFrame(filas, columns=['area', 'perimetro', 'circularidad', 'excentricidad', 'solidez'])

def descriptores_forma(nucleo, roi):
    # Resumen por imagen: mediana de cada descriptor sobre sus núcleos + rel_nc + n_nucleos.
    nuc = medir_nucleos(nucleo)
    if nuc.empty:
        return {k: (0 if k == 'n_nucleos' else np.nan) for k in FORMA}
    d = nuc.median().to_dict()
    d['rel_nc'] = nuc.area.sum() / max(roi.sum(), 1)
    d['n_nucleos'] = len(nuc)
    return d

### 1.2 Ejemplo
Una imagen por subtipo con el contorno del núcleo detectado y sus descriptores, para chequear a ojo que los números tienen sentido.

In [ ]:
subtipos = sorted(org.subtipo.unique())
fig, ax = plt.subplots(1, len(subtipos), figsize=(4 * len(subtipos), 4.6))
for a, st in zip(ax, subtipos):
    r = org[org.subtipo == st].sample(1, random_state=1).iloc[0]
    rgb, roi = preprocesar(leer(r.ruta)), mascara_celula(r.ruta_seg)
    nuc = segmentar(rgb, roi)
    d = descriptores_forma(nuc, roi)
    a.imshow(rgb)
    a.contour(roi, [0.5], colors='yellow', linewidths=1)    # célula
    a.contour(nuc, [0.5], colors='lime', linewidths=1.5)    # núcleo
    a.set_title(f"{st}  ({d['n_nucleos']} núcleos)\nA={d['area']:.0f}  circ={d['circularidad']:.2f}\nsol={d['solidez']:.2f}  N/C={d['rel_nc']:.2f}", fontsize=10)
    a.axis('off')
plt.tight_layout()

### 1.3 Cálculo sobre todo el dataset
Una pasada por las 3.256 imágenes (tarda 1–2 minutos en Colab). Guardamos el resultado en la tabla `desc`, que se va completando en las partes siguientes, y también el área de **todos** los componentes (antes de filtrar) para justificar `AREA_MIN_NUC` con un histograma.

In [ ]:
filas, areas_comp = [], []
for r in org.itertuples():
    rgb, roi = preprocesar(leer(r.ruta)), mascara_celula(r.ruta_seg)
    nuc = segmentar(rgb, roi)
    areas_comp += [reg.area for reg in measure.regionprops(measure.label(nuc))]
    filas.append(dict(archivo=r.archivo, subtipo=r.subtipo, clase=r.clase, roi_frac=r.roi_frac,
                      **descriptores_forma(nuc, roi)))
desc = pd.DataFrame(filas)
print(f"Imágenes sin núcleo detectado: {(desc.n_nucleos == 0).sum()}  (por subtipo: {desc[desc.n_nucleos == 0].subtipo.value_counts().to_dict()})")
desc.groupby('subtipo')[FORMA].median().round(3)

In [ ]:
# ¿Dónde cortar entre "resto/ruido" y "núcleo"? Histograma del área de todos los componentes
plt.figure(figsize=(8, 3.5))
plt.hist(areas_comp, bins=np.logspace(np.log10(25), np.log10(max(areas_comp)), 60))
plt.axvline(AREA_MIN_NUC, color='r', ls='--', label=f'AREA_MIN_NUC = {AREA_MIN_NUC} px²')
plt.xscale('log'); plt.xlabel('Área del componente [px²]'); plt.ylabel('Cantidad'); plt.legend()
plt.title('Área de los componentes de la máscara de núcleo (todas las imágenes)')
plt.tight_layout()

### 1.4 Comparación entre clases
Boxplots de cada descriptor por subtipo. Lo que buscamos son descriptores donde la caja de **Benign** quede separada de las de los subtipos de LLA: esos son los candidatos para las reglas de clasificación del Módulo 4 (donde se evalúa formalmente con Mann-Whitney y curvas ROC).

In [ ]:
fig, ax = plt.subplots(1, len(FORMA), figsize=(3 * len(FORMA), 3.8))
for a, var in zip(ax, FORMA):
    a.boxplot([desc.loc[desc.subtipo == st, var].dropna() for st in subtipos], showfliers=False)
    a.set_xticks(range(1, len(subtipos) + 1), subtipos, rotation=45)
    a.set_title(var)
plt.tight_layout()

In [ ]:
# Mismo resumen pero en la clasificación binaria del proyecto (mediana por clase)
desc.groupby('clase')[FORMA].median().round(3).T

### 1.5 Observaciones (parte 1)
*(Completar después de correr con los resultados reales.)*
- ¿Qué descriptores de forma separan mejor benignas de patológicas?
- ¿Se cumple lo esperado (núcleo más grande, menos circular, menos sólido y mayor N/C en LLA)?
- **Ojo con Pro-B:** si la tabla de 1.0 confirma que muchas imágenes de Pro-B tienen la referencia vacía, sus descriptores reflejan un **error de la ROI y no la biología**. Hay que resolverlo antes de clasificar: una opción es segmentar las células sin depender de la referencia, que además sirve para los casos incógnita de la cátedra.

In [ ]:
desc.to_csv('modulo3_descriptores.csv', index=False)